# zit_only_pearson — fit (best params + seed sweep, iso/tail calibration)

`zit_only_pearson/hpo.py` 병렬 HPO가 만든 Optuna study에서 best trial을 로드해 seed sweep + 후처리
(tau_pi → unit **mean** 집계 → zero_clip) + isotonic/tail calibration을 적용하고 산출물 번들을 저장한다.

- 모델: `ZITboostRegressor` (cell4 MODEL_CLASS). 전처리: `zit_pp.load_for_fit`(cleaning.py 정본, 전 트랙 공용 PP).
- 처리 로직(cell6~15)는 **4조합 공통(byte 동일)** — cell0(제목)·cell4(config)만 조합별로 다르다.
- 산출물: §5.1 의미 폴더 `4_output/01_zit/zit_only_pearson/`.

## 0. 환경 설정


In [ ]:
from pathlib import Path
import gc
import hashlib
import itertools
import json
import os
import pickle
import runpy
import sys
import time
from datetime import datetime

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import os, sys
RESUME = True   # 기존 Optuna study(db)에 이어서 학습할지 (필요시 config 셀에서 덮어씀)
# ── Colab이면 코드 번들 1개(code.zip)만 받아 풀기 — 데이터·경로·폰트는 setup.py가 처리 ──
try:
    import google.colab  # Colab에서만 import 성공
    GDRIVE_CODE_ID = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py+requirements+utils+2_preprocessing+3_modeling 지원코드
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip -q install gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
    os.chdir('/content/project')
except ImportError:
    pass
# ── 공통: cwd에서 위로 setup.py(+utils/)를 자동탐색해 실행 (노트북 깊이·드라이브 위치 무관) ──
_d = os.getcwd()
while not (os.path.exists(os.path.join(_d, 'setup.py')) and os.path.isdir(os.path.join(_d, 'utils'))):
    _p = os.path.dirname(_d)
    if _p == _d:
        raise RuntimeError('프로젝트 루트(setup.py + utils/)를 못 찾음 — cwd 확인')
    _d = _p
if _d not in sys.path:
    sys.path.insert(0, _d)
import runpy
runpy.run_path(os.path.join(_d, 'setup.py'))

from utils.config import (  # setup.py 실행 뒤 import하므로 noqa 유지
    PROJECT_ROOT as CFG_PROJECT_ROOT,
    OUTPUT_DIR,
    TARGET_COL,
    KEY_COL,
    DIE_KEY_COL,
    SEED as DEFAULT_SEED,
)
from utils.data import load_all, get_feat_cols, split_xs  # setup.py 실행 뒤 import하므로 noqa 유지

PP_DIR = Path(CFG_PROJECT_ROOT) / '2_preprocessing'
if str(PP_DIR) not in sys.path:
    sys.path.insert(0, str(PP_DIR))

MOD_DIR = Path(CFG_PROJECT_ROOT) / '3_modeling'
if str(MOD_DIR) not in sys.path:
    sys.path.insert(0, str(MOD_DIR))

from meta_features import add_meta_features  # setup.py 실행 뒤 import하므로 noqa 유지
from modules import preprocess, postprocess  # setup.py 실행 뒤 import하므로 noqa 유지
from sklearn.isotonic import IsotonicRegression  # setup.py 실행 뒤 import하므로 noqa 유지
from sklearn.model_selection import KFold  # setup.py 실행 뒤 import하므로 noqa 유지
from scipy.interpolate import PchipInterpolator  # PCHIP smoothing용

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

PROJECT_ROOT = Path(CFG_PROJECT_ROOT)
OUTPUT_DIR = Path(OUTPUT_DIR)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'OUTPUT_DIR   = {OUTPUT_DIR}')

## 1. 실행 설정


In [ ]:
# 소스: 새 zit_only_pearson Optuna study의 best trial을 db에서 직접 로드한다.
from modules.zit import ZITboostRegressor as MODEL_CLASS
USE_UNIT_ID = False      # bag=True(die에 unit_y 배분), zit_only=False
MODEL_NAME = 'zitboost'   # 산출물 model_name
SRC_NUM = 'zit_only_pearson'           # provenance exp_id 라벨 (cell9)

SOURCE_STUDY_NAME = 'zit_only_pearson'
SOURCE_DB_DIR = Path(OUTPUT_DIR) / '01_zit' / 'zit_only_pearson'
SOURCE_DB_PATH = SOURCE_DB_DIR / f'optuna_jh_{SOURCE_STUDY_NAME}.db'
SOURCE_BEST_DIR = SOURCE_DB_PATH  # provenance(source_best_dir) 기록용

# 산출물 위치 (§5.1 의미 폴더, 실험번호 없음).
OUT_DIR = Path(OUTPUT_DIR) / '01_zit' / 'zit_only_pearson'
BEST_DIR = OUT_DIR / 'best'
SUMMARY_PATH = OUT_DIR / 'seed_sweep_summary.csv'
RESUME = False

# seed pool (4조합 공통).
SEEDS = list(range(1000, 1030))
N_FOLDS = 5
N_JOBS = 6

# 저장 정책. fold_models.pkl 하나가 수십 MB라 기본은 best만 저장한다.
SAVE_EVERY_SEED = False
SAVE_BEST = True

# die->unit 집계. zit_only=mean(모든 후보 탐색), bag=sum(단일 — die가 unit_y/n_die 몫 학습).
BASELINE_AGG = 'mean'
AGG_CANDIDATES = postprocess.AGG_METHODS
POSITION_METHOD = 'optuna'        # AGG_CANDIDATES에 'weighted' 없으면 실질 no-op (시그니처 호환용)
POSITION_OPTUNA_N_TRIALS = 50

# zero_clip: log-spaced 후보 (0.0001~0.003, 30개).
ZERO_CLIP_RANGE = (0.0001, 0.003)
ZERO_CLIP_N = 30
ZERO_CLIP_LOG_SPACE = True

# isotonic/tail 보정 후보 (4조합 공통 grid).
ISO_KINDS = ['step', 'pchip']
ISO_WEIGHTS = [0.25, 0.5, 0.75, 1.00, 1.25, 1.50]
TAIL_QS = [0.95, 0.975, 0.99]
TAIL_RESID_QS = [0.75, 0.90]
TAIL_GAINS = [0.0, 0.5, 1.0, 1.5, 2.5]
TAIL_POWERS = [1.0, 2.0]
IQR_TOP_KS = [0, 1, 2]
IQR_MARGIN = 1e-6

# 실행 규모 확인용.
N_SEEDS = len(SEEDS)
N_MODEL_FITS = N_SEEDS * N_FOLDS
N_CALIBRATION_CANDIDATES = 1 + (
    len(ISO_KINDS) * len(ISO_WEIGHTS) * len(TAIL_QS) * len(TAIL_RESID_QS)
    * len(TAIL_GAINS) * len(TAIL_POWERS) * len(IQR_TOP_KS)
)

OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_DIR.mkdir(parents=True, exist_ok=True)
print(f'SOURCE_STUDY     = {SOURCE_STUDY_NAME}')
print(f'OUT_DIR          = {OUT_DIR}')
print(f'MODEL_CLASS      = {MODEL_CLASS.__name__}  USE_UNIT_ID={USE_UNIT_ID}')
print(f'BASELINE_AGG     = {BASELINE_AGG}  AGG_CANDIDATES={AGG_CANDIDATES}')
print(f'N_SEEDS          = {N_SEEDS}')
print(f'N_MODEL_FITS     = {N_MODEL_FITS}  # seed {N_SEEDS}개 * {N_FOLDS}-fold')
print(f'N_CAL_CAND/SEED  = {N_CALIBRATION_CANDIDATES}')

## 2. best_params.json(study best) 파라미터와 데이터 로드


In [ ]:
import ast
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

_ZIT_DIR = str(PROJECT_ROOT / '3_modeling' / '01_zit')
if _ZIT_DIR not in sys.path:
    sys.path.insert(0, _ZIT_DIR)
import zit_pp  # cleaning.py 정본 PP (전 트랙 공용)


def _parse_user_attr(v):
    if isinstance(v, str):
        try:
            return ast.literal_eval(v)
        except (ValueError, SyntaxError):
            return v
    return v


# 새 study의 best trial을 db에서 직접 로드한다.
_storage = f'sqlite:///{SOURCE_DB_PATH.as_posix()}'
_study = optuna.load_study(study_name=SOURCE_STUDY_NAME, storage=_storage)
_best = _study.best_trial
_ua = {k: _parse_user_attr(v) for k, v in _study.user_attrs.items()}

_bp = dict(_best.params)
best_tau_pi = float(_bp.pop('tau_pi'))

_clip_attr = _ua.get('clip_y_extreme', True)
if isinstance(_clip_attr, str):
    _clip_attr = _clip_attr.lower() in {'true', '1', 'yes'}
clip_y_extreme = bool(_clip_attr)

source_meta = {
    'exp_id': _ua.get('exp_id', SOURCE_STUDY_NAME),
    'model_name': MODEL_NAME,
    'best_trial_number': _best.number,
    'best_oof_rmse': float(_best.value),
    'best_params_resolved': dict(_bp),
    'best_tau_pi': best_tau_pi,
    'effective_pp_params': dict(zit_pp.PP_FIXED),
    'feature_names': None,
    'n_features': None,
    'unit_ids_hash': None,
    'study_meta': {'CLIP_Y_EXTREME': clip_y_extreme, **_ua},
}

base_model_params = dict(source_meta['best_params_resolved'])
pp_fixed = dict(zit_pp.PP_FIXED)
for k in ['random_state', 'n_jobs', 'verbose', 'device', 'em_tol']:
    base_model_params.pop(k, None)

print('[기준 best]')
print(f'  study       : {SOURCE_STUDY_NAME}')
print(f'  best_trial# : {source_meta.get("best_trial_number")}')
print(f'  best_oof    : {source_meta.get("best_oof_rmse"):.9f}')
print(f'  tau_pi      : {best_tau_pi:.9f}')
print(f'  clip extreme: {clip_y_extreme}')

# 데이터 — 통일 PP(zit_pp.load_for_fit): load_all -> clip -> preprocess.run(PP_FIXED) -> add_meta_features.
_d = zit_pp.load_for_fit(clip_y_extreme=clip_y_extreme)
xs_train, xs_val, xs_test = _d['xs_train'], _d['xs_val'], _d['xs_test']
X_train, X_val, X_test = _d['X_train'], _d['X_val'], _d['X_test']
uid_train_die, uid_val_die, uid_test_die = _d['uid_train_die'], _d['uid_val_die'], _d['uid_test_die']
y_train_unit_s, y_val_unit_s, y_test_unit_s = _d['y_train_unit_s'], _d['y_val_unit_s'], _d['y_test_unit_s']
y_train_die = _d['y_train_die']
ys_input = _d['ys_input']
feat_cols_clean = _d['feat_cols']

unit_ids_hash = hashlib.sha1(','.join(map(str, y_train_unit_s.index)).encode()).hexdigest()
print(f'[data] X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}, '
      f'units train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')

## 3. 공통 함수 import (zit_fit_lib) + 결과 저장

In [ ]:
# 공통 calibration/serialization 1차 함수 — zit_fit_lib(4조합 공유)에서 import.
# (실험 드라이버 fit_one_seed/save_result_artifacts는 데이터 결합이라 아래 셀들에 유지.)
from zit_fit_lib import (
    clip_nonneg, apply_tau_pi, unit_rmse,
    tune_unit_postprocess_train_val, fit_iso_tail_grid,
    json_default, build_die_df, build_unit_output, serializable_calibrator,
)

In [ ]:
def save_result_artifacts(res, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    best_cal = res['calibration']
    pp_res = res['postprocess']

    fold_payload = {
        'fold_models': res['fold_models'],
        'feature_names': feat_cols_clean,
        'model_name': MODEL_NAME,
        'n_folds': N_FOLDS,
        'seed': int(res['seed']),
        'fold_model_seeds': res['fold_model_seeds'],
        'em_history_per_fold': res['em_history_per_fold'],
        'source_best_dir': str(SOURCE_BEST_DIR),
        'calibrator': {
            'record': dict(best_cal['record']),
            'iso_model': best_cal.get('iso_model'),
            'iso_kind': best_cal.get('iso_kind'),
            'pchip_knots': best_cal.get('pchip_knots'),
        },
    }
    with open(target_dir / 'fold_models.pkl', 'wb') as f:
        pickle.dump(fold_payload, f)

    raw_train = best_cal['raw_train']
    raw_val = best_cal['raw_val']
    raw_test = best_cal['raw_test']
    build_unit_output(y_train_unit_s, best_cal['train_pred'], raw_train).to_csv(target_dir / 'oof_unit.csv', index=False)
    build_unit_output(y_val_unit_s, best_cal['val_pred'], raw_val).to_csv(target_dir / 'val_unit.csv', index=False)
    build_unit_output(y_test_unit_s, best_cal['test_pred'], raw_test).to_csv(target_dir / 'test_unit.csv', index=False)

    build_die_df(
        uid_train_die,
        xs_train[DIE_KEY_COL].values,
        xs_train['position'].values,
        res['oof_die_pi'],
        res['oof_die_mu'],
        res['oof_die_pred_raw'],
        res['oof_die_pred_taupi'],
        y_train_unit_s,
    ).to_csv(target_dir / 'oof_die.csv', index=False)
    build_die_df(
        uid_val_die,
        xs_val[DIE_KEY_COL].values,
        xs_val['position'].values,
        res['val_die_pi'],
        res['val_die_mu'],
        res['val_die_pred_raw'],
        res['val_die_pred_taupi'],
        y_val_unit_s,
    ).to_csv(target_dir / 'val_die.csv', index=False)
    build_die_df(
        uid_test_die,
        xs_test[DIE_KEY_COL].values,
        xs_test['position'].values,
        res['test_die_pi'],
        res['test_die_mu'],
        res['test_die_pred_raw'],
        res['test_die_pred_taupi'],
        y_test_unit_s,
    ).to_csv(target_dir / 'test_die.csv', index=False)

    best_cal['candidates'].to_csv(target_dir / 'calibration_candidates.csv', index=False)

    meta = {
        'exp_id': f'{SRC_NUM}-seed-sweep-seed{res["seed"]}',
        'model_name': MODEL_NAME,
        'source_best_dir': str(SOURCE_BEST_DIR),
        'source_exp_id': source_meta.get('exp_id'),
        'seed': int(res['seed']),
        'fold_model_seeds': res['fold_model_seeds'],
        'best_params_resolved': res['best_full_params'],
        'best_tau_pi': best_tau_pi,
        'feature_names': feat_cols_clean,
        'n_features': len(feat_cols_clean),
        'n_folds': N_FOLDS,
        'unit_ids_hash': unit_ids_hash,
        'n_units_train': int(len(y_train_unit_s)),
        'n_units_val': int(len(y_val_unit_s)),
        'n_units_test': int(len(y_test_unit_s)),
        'effective_pp_params': pp_fixed,
        'val_rmse': float(res['summary']['val_rmse']),
        'base_val_rmse': float(res['summary']['base_val_rmse']),
        'test_rmse': float(res['summary']['test_rmse']),
        'base_test_rmse': float(res['summary']['base_test_rmse']),
        'postprocess': {
            'best_agg': pp_res['best_agg'],
            'pos_weights': pp_res['pos_weights'].tolist() if pp_res['pos_weights'] is not None else None,
            'best_zero_clip': pp_res['best_zero_clip'],
            'zero_clip_log_space': pp_res['zero_clip_log_space'],
            'zero_clip_arr': pp_res['zero_clip_arr'].tolist(),
            'position_method': pp_res['position_method'],
            'agg_rmses': {k: float(v) for k, v in pp_res['agg_rmses'].items()},
            'train_rmse': float(pp_res['train_rmse']),
            'val_rmse_final': float(pp_res['val_rmse_final']),
            'decisions': pp_res['decisions'],
        },
        'calibration': serializable_calibrator(best_cal),
        'best_iqr12_candidate': best_cal.get('best_iqr12'),
        'created_at': datetime.now().isoformat(timespec='seconds'),
    }
    with open(target_dir / 'best_params.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False, default=json_default)

    with open(target_dir / 'summary_record.json', 'w', encoding='utf-8') as f:
        json.dump(res['summary'], f, indent=2, ensure_ascii=False, default=json_default)

    print(f'[저장 완료] {target_dir}')


## 4. seed 1개 재학습 함수


In [ ]:
# seed마다 unit-level KFold split을 새로 만든다. 같은 unit의 4 die가 train/valid에 섞이지 않도록 unit ID 기준.
def make_folds(seed):
    unique_units = y_train_unit_s.index.values
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=int(seed))
    return unique_units, list(kf.split(unique_units))


def params_for_seed(seed, fold_idx):
    p = dict(base_model_params)
    fold_seed = int(seed) * 1009 + int(fold_idx)
    p['random_state'] = fold_seed
    p['n_jobs'] = N_JOBS
    p['verbose'] = -1
    p['device'] = 'cpu'
    p['em_tol'] = 1e-7
    return p, fold_seed


def fit_one_seed(seed):
    seed = int(seed)
    unique_units, folds = make_folds(seed)

    n_train_die = len(X_train)
    n_val_die = len(X_val)

    oof_die_pi = np.full(n_train_die, np.nan)
    oof_die_mu = np.full(n_train_die, np.nan)
    oof_die_pred_raw = np.full(n_train_die, np.nan)

    val_die_pi = np.zeros(n_val_die)
    val_die_mu = np.zeros(n_val_die)
    val_die_pred_raw = np.zeros(n_val_die)

    n_test_die = len(X_test)
    test_die_pi = np.zeros(n_test_die)
    test_die_mu = np.zeros(n_test_die)
    test_die_pred_raw = np.zeros(n_test_die)

    fold_models = []
    fold_model_seeds = []
    em_history_per_fold = []

    t0 = time.time()
    print(f'\n=== seed {seed} ===')
    for fold_idx, (tr_uidx, vl_uidx) in enumerate(folds):
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        params, fold_seed = params_for_seed(seed, fold_idx)
        model = MODEL_CLASS(**params)
        if USE_UNIT_ID:
            model.fit(X_train[tr_mask], y_train_die[tr_mask], unit_id=uid_train_die[tr_mask])
        else:
            model.fit(X_train[tr_mask], y_train_die[tr_mask])

        pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
        pred_vl = clip_nonneg((1.0 - pi_vl) * mu_vl)
        oof_die_pi[vl_mask] = pi_vl
        oof_die_mu[vl_mask] = mu_vl
        oof_die_pred_raw[vl_mask] = pred_vl

        pi_val, mu_val, _ = model.predict_components(X_val)
        val_die_pi += pi_val / N_FOLDS
        val_die_mu += mu_val / N_FOLDS
        val_die_pred_raw += clip_nonneg((1.0 - pi_val) * mu_val) / N_FOLDS

        # test도 같은 5-fold 앙상블 평균. test y는 선택에 절대 쓰지 않고 모니터링/제출용 예측만 만든다.
        pi_test, mu_test, _ = model.predict_components(X_test)
        test_die_pi += pi_test / N_FOLDS
        test_die_mu += mu_test / N_FOLDS
        test_die_pred_raw += clip_nonneg((1.0 - pi_test) * mu_test) / N_FOLDS

        fold_models.append(model)
        fold_model_seeds.append(fold_seed)
        em_history_per_fold.append(getattr(model, 'em_history_', None))
        print(f'  fold {fold_idx + 1}/{N_FOLDS} done, model_seed={fold_seed}, elapsed={time.time() - t0:.0f}s')

    if np.isnan(oof_die_pred_raw).any():
        raise RuntimeError(f'seed {seed}: OOF die prediction has NaN')

    oof_die_pred_taupi = apply_tau_pi(oof_die_pred_raw, oof_die_pi, best_tau_pi)
    val_die_pred_taupi = apply_tau_pi(val_die_pred_raw, val_die_pi, best_tau_pi)
    test_die_pred_taupi = apply_tau_pi(test_die_pred_raw, test_die_pi, best_tau_pi)

    pp_res = tune_unit_postprocess_train_val(
        xs_train=xs_train,
        xs_val=xs_val,
        xs_test=xs_test,
        die_pred_train=oof_die_pred_taupi,
        die_pred_val=val_die_pred_taupi,
        die_pred_test=test_die_pred_taupi,
        y_train_unit_df=ys_input['train'],
        y_val_unit_df=ys_input['validation'],
        baseline_agg=BASELINE_AGG, agg_candidates=AGG_CANDIDATES,
        position_method=POSITION_METHOD, position_optuna_n_trials=POSITION_OPTUNA_N_TRIALS,
        zero_clip_range=ZERO_CLIP_RANGE, zero_clip_n=ZERO_CLIP_N,
        zero_clip_log_space=ZERO_CLIP_LOG_SPACE,
    )

    cal = fit_iso_tail_grid(
        pp_res['final_train_unit'], pp_res['final_val_unit'], pp_res['final_test_unit'],
        y_train_unit_s, y_val_unit_s, y_test_unit_s,
        iso_kinds=ISO_KINDS, iso_weights=ISO_WEIGHTS, tail_qs=TAIL_QS, tail_resid_qs=TAIL_RESID_QS,
        tail_gains=TAIL_GAINS, tail_powers=TAIL_POWERS, iqr_top_ks=IQR_TOP_KS, iqr_margin=IQR_MARGIN,
    )
    best_rec = cal['record']
    best_iqr12 = cal.get('best_iqr12')

    # base_test_rmse: postprocess까지만 적용한 test RMSE (calibration 전). base_val_rmse의 짝.
    base_test_rmse = unit_rmse(pp_res['final_test_unit'], y_test_unit_s)

    elapsed = time.time() - t0
    summary = {
        'seed': seed,
        'elapsed_sec': float(elapsed),
        'base_train_rmse': float(pp_res['train_rmse']),
        'base_val_rmse': float(pp_res['val_rmse_final']),
        'base_test_rmse': float(base_test_rmse),
        'val_rmse': float(best_rec['val_rmse']),
        'test_rmse': float(best_rec['test_rmse']),
        'train_rmse': float(best_rec['train_rmse']),
        'calibration_name': best_rec['name'],
        'uses_iso': bool(best_rec['uses_iso']),
        'iso_kind': str(best_rec.get('iso_kind', 'none')),
        'iso_weight': float(best_rec.get('iso_weight', 0.0)),
        'tail_q': float(best_rec.get('tail_q', np.nan)) if not pd.isna(best_rec.get('tail_q', np.nan)) else np.nan,
        'tail_resid_q': float(best_rec.get('tail_resid_q', np.nan)) if not pd.isna(best_rec.get('tail_resid_q', np.nan)) else np.nan,
        'tail_gain': float(best_rec.get('tail_gain', 0.0)),
        'tail_power': float(best_rec.get('tail_power', np.nan)) if not pd.isna(best_rec.get('tail_power', np.nan)) else np.nan,
        'tail_resid_scale': float(best_rec.get('tail_resid_scale', 0.0)),
        'iqr_top_k': int(best_rec.get('iqr_top_k', 0)),
        'val_iqr_outliers': int(best_rec['val_iqr_outliers']),
        'val_iqr_upper_fence': float(best_rec['val_iqr_upper_fence']),
        'val_max_pred': float(best_rec['val_max_pred']),
        'val_outlier_true_mean': float(best_rec['val_outlier_true_mean']) if not pd.isna(best_rec['val_outlier_true_mean']) else np.nan,
        'val_outlier_true_max': float(best_rec['val_outlier_true_max']) if not pd.isna(best_rec['val_outlier_true_max']) else np.nan,
        'val_outlier_true_ge_q95': int(best_rec['val_outlier_true_ge_q95']),
        'val_top_pred_y_true': float(best_rec['val_top_pred_y_true']),
        'best_iqr12_val_rmse': float(best_iqr12['val_rmse']) if best_iqr12 else np.nan,
        'best_iqr12_name': best_iqr12['name'] if best_iqr12 else None,
        'postprocess_best_agg': pp_res['best_agg'],
        'postprocess_best_zero_clip': pp_res['best_zero_clip'],
    }
    print(f'[seed {seed}] base_val={summary["base_val_rmse"]:.9f}, best_val={summary["val_rmse"]:.9f}, '
          f'test={summary["test_rmse"]:.9f}, cal={summary["calibration_name"]}, '
          f'iqr_outliers={summary["val_iqr_outliers"]}, elapsed={elapsed:.0f}s')

    best_full_params, _ = params_for_seed(seed, 0)
    return {
        'seed': seed,
        'summary': summary,
        'postprocess': pp_res,
        'calibration': cal,
        'fold_models': fold_models,
        'fold_model_seeds': fold_model_seeds,
        'em_history_per_fold': em_history_per_fold,
        'best_full_params': best_full_params,
        'oof_die_pi': oof_die_pi,
        'oof_die_mu': oof_die_mu,
        'oof_die_pred_raw': oof_die_pred_raw,
        'oof_die_pred_taupi': oof_die_pred_taupi,
        'val_die_pi': val_die_pi,
        'val_die_mu': val_die_mu,
        'val_die_pred_raw': val_die_pred_raw,
        'val_die_pred_taupi': val_die_pred_taupi,
        'test_die_pi': test_die_pi,
        'test_die_mu': test_die_mu,
        'test_die_pred_raw': test_die_pred_raw,
        'test_die_pred_taupi': test_die_pred_taupi,
    }

## 5. seed sweep 실행


In [ ]:
# RESUME=True이면 기존 summary를 읽고 이미 끝난 seed는 건너뛴다.
if RESUME and SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)
    rows = summary_df.to_dict('records')
    done_seeds = set(summary_df['seed'].astype(int).tolist())
    best_so_far = float(summary_df['val_rmse'].min()) if len(summary_df) else float('inf')
    print(f'[재개] 기존 {len(summary_df)}개 seed 로드, 현재 best={best_so_far:.9f}')
else:
    rows = []
    done_seeds = set()
    best_so_far = float('inf')


# IQR 상단 이상치 점검은 성능(RMSE)과 분리한다.
#   - 성능: fit_one_seed가 train/val/test 각각 따로 RMSE를 계산한다 (concat 안 함).
#   - 이상치: best calibration 후보 예측을 train+val+test로 concat한 전체 분포에서 IQRx1.5 상단 이상치 유무/개수만 본다.
# iqr_stats와 동일한 fence 정의(q3 + 1.5*IQR, 초과분)를 써서 split별 지표와 기준을 맞춘다.
def concat_iqr_outlier_stats(cal):
    pred = np.concatenate([cal['train_pred'], cal['val_pred'], cal['test_pred']])
    q1, q3 = np.quantile(pred, [0.25, 0.75])
    upper_fence = float(q3 + 1.5 * (q3 - q1))
    n_out = int((pred > upper_fence).sum())
    return {
        'concat_n_total': int(len(pred)),
        'concat_iqr_upper_fence': upper_fence,
        'concat_iqr_outliers': n_out,
        'concat_has_outlier': bool(n_out > 0),
        'concat_max_pred': float(pred.max()),
    }


for seed in SEEDS:
    if int(seed) in done_seeds:
        print(f'[건너뜀] seed {seed}는 이미 summary에 있음')
        continue

    res = fit_one_seed(seed)

    # 성능과 별개로, concat(train+val+test) 분포 기준 IQR 상단 이상치 정보를 summary에 덧붙인다.
    concat_stats = concat_iqr_outlier_stats(res['calibration'])
    res['summary'].update(concat_stats)
    print(f"  [concat IQR] train+val+test n={concat_stats['concat_n_total']}, "
          f"upper_fence={concat_stats['concat_iqr_upper_fence']:.6f}, "
          f"상단 이상치={concat_stats['concat_iqr_outliers']}개, "
          f"max={concat_stats['concat_max_pred']:.6f}")

    rows.append(res['summary'])
    summary_df = pd.DataFrame(rows).sort_values('val_rmse').reset_index(drop=True)
    summary_df.to_csv(SUMMARY_PATH, index=False)

    seed_is_best = res['summary']['val_rmse'] < best_so_far
    if SAVE_EVERY_SEED:
        save_result_artifacts(res, OUT_DIR / 'seeds' / f'seed_{int(seed)}')
    if SAVE_BEST and seed_is_best:
        best_so_far = float(res['summary']['val_rmse'])
        save_result_artifacts(res, BEST_DIR)
        print(f'[새 best] seed={seed}, val_rmse={best_so_far:.9f}')

    del res
    gc.collect()

print('\n[완료]')
display(pd.read_csv(SUMMARY_PATH).sort_values('val_rmse').head(20))
print(f'best 산출물: {BEST_DIR}')

## 6. 결과 확인


In [ ]:
summary = pd.read_csv(SUMMARY_PATH).sort_values('val_rmse').reset_index(drop=True)
display(summary.head(30))

# isotonic kind별 best 분포 확인 - step vs pchip 중 어느 쪽이 더 자주 best가 됐는지.
if 'iso_kind' in summary.columns:
    print('\n[iso_kind 분포]')
    display(summary['iso_kind'].value_counts())
    print('\n[iso_kind별 평균 val_rmse]')
    display(summary.groupby('iso_kind')['val_rmse'].agg(['count', 'mean', 'min']))

# val_rmse는 다수 후보 중 best를 val로 골라서 낙관적(selection bias). test_rmse가 정직한 일반화 지표.
if 'test_rmse' in summary.columns:
    print('\n[val vs test 갭 - 클수록 val 과적합 신호]')
    gap = summary['test_rmse'] - summary['val_rmse']
    print(f'  mean val_rmse  = {summary["val_rmse"].mean():.9f}')
    print(f'  mean test_rmse = {summary["test_rmse"].mean():.9f}')
    print(f'  mean gap(test-val) = {gap.mean():.9f}')

# IQR 상단 이상치는 성능과 분리해 train+val+test를 concat한 전체 분포 기준으로 본다.
if 'concat_iqr_outliers' in summary.columns:
    print()
    print('[concat(train+val+test) IQRx1.5 상단 이상치]')
    n_seed = len(summary)
    n_have = int((summary['concat_iqr_outliers'] > 0).sum())
    print(f'  이상치 보유 seed = {n_have}/{n_seed}')
    print(f'  seed별 이상치 개수: min={int(summary["concat_iqr_outliers"].min())}, '
          f'median={summary["concat_iqr_outliers"].median():.1f}, '
          f'max={int(summary["concat_iqr_outliers"].max())}')
    _ccols = [c for c in ['seed', 'val_rmse', 'test_rmse', 'concat_iqr_outliers',
                          'concat_iqr_upper_fence', 'concat_max_pred', 'concat_n_total']
              if c in summary.columns]
    display(summary[_ccols].sort_values('val_rmse').head(30))

iqr12 = summary[summary['val_iqr_outliers'].between(1, 2)].copy()
print('\n[validation RMSE 기준 best]')
display(summary.head(1))

print('\n[IQR upper outlier 1~2개 조건 best]')
if len(iqr12):
    display(iqr12.sort_values('val_rmse').head(10))
else:
    print('아직 IQR upper outlier 1~2개 조건을 만족하는 후보가 없습니다.')

print(f'OUT_DIR  = {OUT_DIR}')
print(f'BEST_DIR = {BEST_DIR}')